# Phase III: BERT Fine-tuned POS Tagger
This notebook fine-tunes `prajjwal1/bert-mini` on the `conll2003` dataset for Part-of-Speech (POS) tagging.
We use the `transformers` and `datasets` libraries, and evaluate using the `seqeval` metric.


In [ ]:
# Install required libraries
!pip install seqeval evaluate datasets transformers

In [ ]:
import warnings
# Suppress UserWarnings from seqeval and transformers
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*seems not to be NE tag.*")

In [ ]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

# Device setup
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [ ]:
from datasets import load_dataset
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")


print(dataset)
print("\nFirst training example:")
print(dataset["train"][0])

# Accessing the feature names correctly
pos_feature = dataset["train"].features["pos_tags"]
if hasattr(pos_feature, 'feature'):
    label_names = pos_feature.feature.names
else:
    label_names = pos_feature.names

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

num_labels = len(label_names)
print(f"\nNumber of POS tags: {num_labels}")
print(f"Tags: {label_names}")

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

First training example:
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

Number of POS tags: 47
Tags: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


## 2. Tokenization and Alignment
We load the tokenizer for `prajjwal1/bert-mini`.
We need to align the POS tags to the subword tokens. The first subword token of each word gets the original POS tag, while subsequent subwords (and special tokens) get `-100` so they are ignored by the loss function.


In [ ]:
model_checkpoint = "prajjwal1/bert-mini"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=128)

    labels = []
    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens mapped to None
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Other tokens in a word get -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply tokenization
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


## 3. Metrics Definition
We use `seqeval` for evaluation which gives us token-level accuracy, and macro F1 score.


In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [
        [label_names[l] for l in label if l != -100] for label in labels
    ]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


## 4. Model Setup and Training
We use `AutoModelForTokenClassification` and prepare `TrainingArguments`.
For time and compute considerations, we'll train for a small number of epochs or steps.


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="./bert-mini-pos",
    eval_strategy="epoch",
    learning_rate=0.00002,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=20,
    weight_decay=0.01,
    push_to_hub=False,
    logging_steps=100,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 5. Training
Run the training hook.


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.890756,0.684908,0.840243,0.820672,0.830342,0.879950
2,0.543374,0.457417,0.874557,0.862356,0.868414,0.905533
3,0.401080,0.387041,0.885482,0.875191,0.880307,0.913964
4,0.356553,0.349507,0.893875,0.880602,0.887189,0.919220
5,0.320903,0.328667,0.897800,0.887104,0.892420,0.923153
6,0.301389,0.319730,0.897971,0.887056,0.892480,0.923251
7,0.285560,0.308416,0.901798,0.891011,0.896372,0.926054
8,0.265109,0.303603,0.902505,0.893243,0.897850,0.926853
9,0.259252,0.299294,0.903207,0.892491,0.897817,0.927008
10,0.250745,0.295789,0.904919,0.893194,0.899018,0.927865


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8780, training_loss=0.35086440410049197, metrics={'train_runtime': 289.0279, 'train_samples_per_second': 971.602, 'train_steps_per_second': 30.378, 'total_flos': 264264640054470.0, 'train_loss': 0.35086440410049197, 'epoch': 20.0})

## 6. Evaluation and Inference
Evaluate on the test set and try a sample sentence.


In [ ]:
print("\nEvaluating on test set:")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)



Evaluating on test set:


{'eval_loss': 0.29063865542411804, 'eval_precision': 0.903405537551879, 'eval_recall': 0.8879823409427477, 'eval_f1': 0.8956275451970634, 'eval_accuracy': 0.9296005168515129, 'eval_runtime': 4.0433, 'eval_samples_per_second': 854.007, 'eval_steps_per_second': 26.711, 'epoch': 20.0}


In [ ]:
from transformers import pipeline

pos_pipeline = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=0 if device != "cpu" else -1)

text = "Michael Scott is the regional manager of Dunder Mifflin Paper Company."
print(f"\nInput text: {text}")
results = pos_pipeline(text)
for res in results:
    print(f"Word: {res['word']:<15} | POS: {res['entity_group']}")



Input text: Michael Scott is the regional manager of Dunder Mifflin Paper Company.
Word: Michael Scott   | POS: NNP
Word: is              | POS: VBZ
Word: the             | POS: DT
Word: regional        | POS: JJ
Word: manager         | POS: NN
Word: of              | POS: IN
Word: Dunder Mifflin Paper Company | POS: NNP
Word: .               | POS: .


In [ ]:
import pandas as pd

# Get predictions from the test set
predictions_output = trainer.predict(tokenized_datasets["test"])

# Pass the (logits, labels) tuple to our custom compute_metrics function
final_metrics = compute_metrics((predictions_output.predictions, predictions_output.label_ids))

# Display the results in a clean table
metrics_df = pd.DataFrame([final_metrics])
display(metrics_df)

,precision,recall,f1,accuracy
0,0.903406,0.887982,0.895628,0.929601
